In [5]:
import numpy as np
import os
import struct
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import optuna

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load Data
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Standard Scaler
print("\n--- Applying StandardScaler ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. OPTUNA Optimization for Neural Network
print("\n--- Starting Optuna optimization for Neural Network (MLP) ---")
def objective(trial):
    hidden_size = trial.suggest_int('hidden_size', 16, 32)
    alpha = trial.suggest_float('alpha', 1e-4, 1e-1, log=True)
    
    mlp = MLPClassifier(hidden_layer_sizes=(hidden_size,), activation='relu', 
                        solver='adam', alpha=alpha, max_iter=2000, random_state=42)
    mlp.fit(X_train_scaled, y_train)
    preds = mlp.predict(X_test_scaled)
    return accuracy_score(y_test, preds)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")
print(f"Best Parameters: {study.best_params}")

# 4. Train Final Model
print("\n--- Training Final Lightweight Neural Network ---")
final_mlp = MLPClassifier(hidden_layer_sizes=(study.best_params['hidden_size'],), 
                          activation='relu', solver='adam', 
                          alpha=study.best_params['alpha'], max_iter=5000, random_state=42)
final_mlp.fit(X_train_scaled, y_train)
preds = final_mlp.predict(X_test_scaled)

target_names = ['on', 'off', 'stop', 'unknown']
print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. EXPORT NEURAL NETWORK AS BINARY (ZERO-RAM PARSING)
print("\n--- Exporting Neural Network as Binary ---")
output_file = os.path.join(OUTPUT_PATH, 'model_weights.bin')

with open(output_file, 'wb') as f:
    # 1. نوشتن ابعاد شبکه (تعداد ویژگی‌ها، نورون‌ها و کلاس‌ها)
    f.write(struct.pack('<HHH', 39, study.best_params['hidden_size'], 4))
    
    # 2. نوشتن پارامترهای نرمال‌ساز
    for val in scaler.mean_: f.write(struct.pack('<f', float(val)))
    for val in scaler.scale_: f.write(struct.pack('<f', float(val)))
    
    # 3. نوشتن لایه اول (W1, B1)
    for row in final_mlp.coefs_[0]:
        for val in row: f.write(struct.pack('<f', float(val)))
    for val in final_mlp.intercepts_[0]:
        f.write(struct.pack('<f', float(val)))
        
    # 4. نوشتن لایه خروجی (W2, B2)
    for row in final_mlp.coefs_[1]:
        for val in row: f.write(struct.pack('<f', float(val)))
    for val in final_mlp.intercepts_[1]:
        f.write(struct.pack('<f', float(val)))

print(f"Neural Network successfully exported to: {output_file}")
print("!!! ACTION REQUIRED: Move 'model_weights.bin' to your SD Card !!!")
# Extract Weights (W) and Biases (B)
W1 = final_mlp.coefs_[0].tolist()       # Input to Hidden Weights
B1 = final_mlp.intercepts_[0].tolist()  # Hidden Biases
W2 = final_mlp.coefs_[1].tolist()       # Hidden to Output Weights
B2 = final_mlp.intercepts_[1].tolist()  # Output Biases

with open(output_file, 'w') as f:
    f.write("# Auto-Generated Edge Neural Network (MLP)\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    f.write(f"SCALER_MEAN = {[round(float(v), 6) for v in scaler.mean_]}\n")
    f.write(f"SCALER_SCALE = {[round(float(v), 6) for v in scaler.scale_]}\n\n")
    
    f.write("W1 = [\n")
    for row in W1: f.write(f"    {[round(float(w), 6) for w in row]},\n")
    f.write("]\n\n")
    
    f.write(f"B1 = {[round(float(b), 6) for b in B1]}\n\n")
    
    f.write("W2 = [\n")
    for row in W2: f.write(f"    {[round(float(w), 6) for w in row]},\n")
    f.write("]\n\n")
    
    f.write(f"B2 = {[round(float(b), 6) for b in B2]}\n")

print(f"Neural Network successfully exported to: {output_file}")

Loaded Data -> X shape: (6000, 39), y shape: (6000,)

--- Applying StandardScaler ---

--- Starting Optuna optimization for Neural Network (MLP) ---
Best Accuracy during CV: 73.42%
Best Parameters: {'hidden_size': 31, 'alpha': 0.0001131608144580149}

--- Training Final Lightweight Neural Network ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.73      0.80      0.76       300
         off       0.69      0.68      0.69       300
        stop       0.88      0.82      0.85       300
     unknown       0.65      0.64      0.64       300

    accuracy                           0.73      1200
   macro avg       0.74      0.73      0.73      1200
weighted avg       0.74      0.73      0.73      1200


--- Exporting Neural Network as Binary ---
Neural Network successfully exported to: ../edge_mcu\model_weights.bin
!!! ACTION REQUIRED: Move 'model_weights.bin' to your SD Card !!!
Neural Network successfully exported to: ../edge_mcu\

In [6]:
import struct
import os

OUTPUT_PATH = '../edge_mcu' # مسیر خروجی خود را چک کنید که درست باشد
output_file = os.path.join(OUTPUT_PATH, 'model_weights.bin')

with open(output_file, 'wb') as f:
    # 1. نوشتن هدر (3 عدد صحیح بدون علامت)
    f.write(struct.pack('<HHH', 39, final_mlp.hidden_layer_sizes[0], 4))
    
    # 2. پارامترهای نرمال‌ساز (اعداد اعشاری 32 بیتی)
    for val in scaler.mean_: f.write(struct.pack('<f', float(val)))
    for val in scaler.scale_: f.write(struct.pack('<f', float(val)))
    
    # 3. لایه مخفی شبکه عصبی
    for row in final_mlp.coefs_[0]:
        for val in row: f.write(struct.pack('<f', float(val)))
    for val in final_mlp.intercepts_[0]:
        f.write(struct.pack('<f', float(val)))
        
    # 4. لایه خروجی شبکه عصبی
    for row in final_mlp.coefs_[1]:
        for val in row: f.write(struct.pack('<f', float(val)))
    for val in final_mlp.intercepts_[1]:
        f.write(struct.pack('<f', float(val)))

print(f"✅ Binary Neural Network created successfully!")
print(f"File saved at: {output_file}")
print("-> Now copy THIS file to your SD Card.")

✅ Binary Neural Network created successfully!
File saved at: ../edge_mcu\model_weights.bin
-> Now copy THIS file to your SD Card.
